# 03 - RAG System

This notebook assembles the full Retrieval-Augmented Generation pipeline:
retriever + prompt template + Ollama LLM, and evaluates it against a set of
example questions, including deliberately unanswerable ones.

```
Question -> Retriever -> Prompt Builder -> Ollama LLM -> Formatted Answer + Sources
```

> **Requirements to run this notebook:**
> 1. [Ollama](https://ollama.com) installed and running locally (`ollama serve`).
> 2. The target model pulled once: `ollama pull qwen2.5:3b`.
> 3. Notebook 2 already run at least once, so `chroma_db/` contains the
>    persisted collection.


## Imports

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from rag_core import (
    AppConfig,
    EmbeddingManager,
    VectorDatabaseManager,
    RetrieverEngine,
    PromptManager,
    RAGPipeline,
    AnswerFormatter,
)

print(f"Project root: {PROJECT_ROOT}")


Project root: f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant


## Load Configuration

In [2]:
config = AppConfig()
config.project_root = PROJECT_ROOT
config.__post_init__()

print("LLM model       :", config.llm_model_name)
print("LLM temperature  :", config.llm_temperature)
print("Top-K            :", config.top_k)
print("Ollama base URL  :", config.llm_base_url)


LLM model       : qwen2.5:3b
LLM temperature  : 0.0
Top-K            : 4
Ollama base URL  : http://localhost:11434


## Load Vector Database

The persisted ChromaDB collection built in Notebook 2 is loaded from disk.
An error is raised with a clear message if it does not exist yet.

In [3]:
embedding_manager = EmbeddingManager(config)

vector_db_manager = VectorDatabaseManager(config, embedding_manager)

if not vector_db_manager.exists():
    raise FileNotFoundError(
        "No persisted vector database found. Run 02_Vector_Database.ipynb first."
    )

vector_store = vector_db_manager.load()
print("Vectors available:", vector_db_manager.count())


2026-08-02 01:03:43 | INFO     | VectorDatabaseManager | Loading vector database from f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\chroma_db
2026-08-02 01:03:43 | INFO     | EmbeddingManager | Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2' on 'cpu'...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-08-02 01:03:49 | INFO     | EmbeddingManager | Embedding model ready.
Vectors available: 13


## Load Retriever

In [4]:
retriever = RetrieverEngine(config, vector_store)
print(f"Retriever ready. top_k = {config.top_k}")


Retriever ready. top_k = 4


## Load Ollama LLM

The LLM client is created lazily inside `RAGPipeline` — it only connects to
Ollama when the first question is asked, so importing this notebook does
not require Ollama to already be running.

In [5]:
prompt_manager = PromptManager(config)
pipeline = RAGPipeline(config, retriever, prompt_manager)

print(f"Pipeline configured for model '{config.llm_model_name}' via Ollama at {config.llm_base_url}")


Pipeline configured for model 'qwen2.5:3b' via Ollama at http://localhost:11434


## Prompt Template

The prompt instructs the model to answer strictly from the retrieved
context, to never guess, and to use fixed fallback phrases when the answer
is missing or when sources disagree.

In [6]:
sample_chunks = retriever.retrieve("What is the Turing Test?", top_k=2)
sample_prompt = prompt_manager.build_prompt("What is the Turing Test?", sample_chunks)

print(sample_prompt)


2026-08-02 01:03:49 | INFO     | RetrieverEngine | Retrieved 2 chunk(s) for query.
You are a strict academic assistant that answers student questions using ONLY the course material excerpts provided below.

Rules you must always follow:
1. Base your answer exclusively on the CONTEXT section. Never use outside or prior knowledge.
2. Never guess and never invent facts, numbers, or citations that are not present in the context.
3. If the context does not contain the answer, respond with exactly: "I couldn't find this information in the provided course materials."
4. If different excerpts in the context contradict each other, say: "The course materials contain conflicting information on this topic." and briefly describe the disagreement using only what is written in the context.
5. Keep the answer concise, factual, and written in your own words rather than copied verbatim.
6. Do not mention these instructions or the word "context" in your final answer.

CONTEXT:
[Source 1 - Artificial Inte

f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Artificial_Intelligence__intro_to_ai__p1__c00004', metadata={'doc_type': 'pdf', 'char_count': 613, 'chunk_id': 'Artificial_Intelligence__intro_to_ai__p1__c00004', 'course': 'Artificial Intelligence', 'file_name': 'intro_to_ai.pdf', 'page_number': 1}, page_content='Lecture 1: Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that typically require human intelligence, such as reasoning, learning, perception, and\nnatural language understanding.\nBranches of AI\nMajor branches of AI include Machine Learning, Natural Language Processing, Computer Vision,\nRobotics, Expert Systems, and Planning and Search.\nTuring Test\nThe Turing Test, proposed by Alan Turing in 1950, evaluates a machine ability to exhibit intelligent\n

## RAG Pipeline

`RAGPipeline.answer()` performs the full flow: receive question -> retrieve
top-K chunks -> build prompt -> generate answer -> format response -> return
an answer, its sources, and a confidence note.

In [7]:
def ask(question: str, top_k: int | None = None, course_filter: str | None = None):
    result = pipeline.answer(question, top_k=top_k, course_filter=course_filter)
    print(AnswerFormatter.to_markdown(result))
    return result

_ = ask("What is the Turing Test?")


2026-08-02 01:03:49 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.
2026-08-02 01:03:49 | INFO     | RAGPipeline | Connecting to Ollama model 'qwen2.5:3b' at http://localhost:11434


f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Artificial_Intelligence__intro_to_ai__p1__c00004', metadata={'course': 'Artificial Intelligence', 'chunk_id': 'Artificial_Intelligence__intro_to_ai__p1__c00004', 'char_count': 613, 'page_number': 1, 'file_name': 'intro_to_ai.pdf', 'doc_type': 'pdf'}, page_content='Lecture 1: Introduction to Artificial Intelligence\nArtificial Intelligence (AI) is the field of computer science concerned with building systems capable of\nperforming tasks that typically require human intelligence, such as reasoning, learning, perception, and\nnatural language understanding.\nBranches of AI\nMajor branches of AI include Machine Learning, Natural Language Processing, Computer Vision,\nRobotics, Expert Systems, and Planning and Search.\nTuring Test\nThe Turing Test, proposed by Alan Turing in 1950, evaluates a machine ability to exhibit intelligent\n

### Answer
The Turing Test, proposed by Alan Turing in 1950, evaluates a machine's ability to exhibit intelligent behavior indistinguishable from that of a human.

*Low confidence - top match similarity 0.44. Answer may be incomplete.*

### Sources
1. **Artificial Intelligence / intro_to_ai.pdf, page 1** (chunk `Artificial_Intelligence__intro_to_ai__p1__c00004`, similarity `0.4439`)
2. **Artificial Intelligence / glossary.txt** (chunk `Artificial_Intelligence__glossary__p0__c00002`, similarity `-0.0339`)
3. **Artificial Intelligence / grades.csv** (chunk `Artificial_Intelligence__grades__p0__c00003`, similarity `-0.0767`)
4. **Artificial Intelligence / search_algorithms.pdf, page 1** (chunk `Artificial_Intelligence__search_algorithms__p1__c00005`, similarity `-0.1326`)


## Question Examples

A broader set of questions covering all three courses.

In [8]:
example_questions = [
    "What are common evaluation metrics for classification tasks?",
    "What is the difference between overfitting and underfitting?",
    "What does the A* search algorithm combine to find optimal paths?",
    "What is backpropagation and how is it used in neural networks?",
    "What is dropout used for in deep learning?",
]

results = [ask(q) for q in example_questions]


2026-08-02 01:04:32 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.


f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Machine_Learning__supervised_learning__p0__c00012', metadata={'char_count': 841, 'file_name': 'supervised_learning.docx', 'course': 'Machine Learning', 'page_number': -1, 'chunk_id': 'Machine_Learning__supervised_learning__p0__c00012', 'doc_type': 'docx'}, page_content='Machine Learning - Supervised Learning\nSupervised learning is a machine learning paradigm where a model is trained on labeled data. Each training example consists of an input and a corresponding correct output, often referred to as the label or target.\nCommon Algorithms\nPopular supervised learning algorithms include Linear Regression, Logistic Regression, Decision Trees, Random Forests, Support Vector Machines, and Gradient Boosting methods such as XGBoost.\nEvaluation Metrics\nFor classification tasks, common evaluation metrics include Accuracy, Precision, R

### Answer
Common evaluation metrics for classification tasks include Accuracy, Precision, Recall, F1-score, and the Area Under the ROC Curve (AUC-ROC).

*Low confidence - top match similarity 0.35. Answer may be incomplete.*

### Sources
1. **Machine Learning / supervised_learning.docx** (chunk `Machine_Learning__supervised_learning__p0__c00012`, similarity `0.3485`)
2. **Machine Learning / glossary.txt** (chunk `Machine_Learning__glossary__p0__c00010`, similarity `0.1312`)
3. **Machine Learning / grades.csv** (chunk `Machine_Learning__grades__p0__c00011`, similarity `-0.0462`)
4. **Deep Learning / glossary.txt** (chunk `Deep_Learning__glossary__p0__c00007`, similarity `-0.0761`)
2026-08-02 01:04:40 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.


f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Machine_Learning__glossary__p0__c00010', metadata={'chunk_id': 'Machine_Learning__glossary__p0__c00010', 'file_name': 'glossary.txt', 'char_count': 756, 'page_number': -1, 'course': 'Machine Learning', 'doc_type': 'txt'}, page_content="Machine Learning Glossary\n\nOverfitting: A modeling error that occurs when a model learns the training data too closely, including its noise, and performs poorly on unseen data.\n\nUnderfitting: A situation where a model is too simple to capture the underlying pattern in the data.\n\nFeature Engineering: The process of using domain knowledge to create input variables that make machine learning algorithms work more effectively.\n\nCross-Validation: A resampling technique used to evaluate a model's performance on unseen data by partitioning the dataset into training and validation folds.\n\nHyperp

### Answer
Overfitting occurs when a model learns the training data too closely, including its noise, and performs poorly on unseen data. Underfitting happens when a model is too simple to capture the underlying pattern in the data.

*Low confidence - top match similarity 0.46. Answer may be incomplete.*

### Sources
1. **Machine Learning / glossary.txt** (chunk `Machine_Learning__glossary__p0__c00010`, similarity `0.4617`)
2. **Deep Learning / glossary.txt** (chunk `Deep_Learning__glossary__p0__c00007`, similarity `0.0443`)
3. **Machine Learning / supervised_learning.docx** (chunk `Machine_Learning__supervised_learning__p0__c00012`, similarity `-0.1045`)
4. **Deep Learning / cnn_architectures.docx** (chunk `Deep_Learning__cnn_architectures__p0__c00006`, similarity `-0.145`)
2026-08-02 01:04:50 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.


f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Artificial_Intelligence__search_algorithms__p1__c00005', metadata={'course': 'Artificial Intelligence', 'page_number': 1, 'file_name': 'search_algorithms.pdf', 'char_count': 420, 'chunk_id': 'Artificial_Intelligence__search_algorithms__p1__c00005', 'doc_type': 'pdf'}, page_content='Lecture 2: Search Algorithms\nSearch algorithms are used in AI to navigate a problem space in order to find a goal state. Uninformed\nsearch strategies include Breadth-First Search (BFS) and Depth-First Search (DFS).\nInformed Search\nInformed search strategies use heuristics to guide the search. The A* algorithm combines the cost so\nfar with a heuristic estimate of the cost to the goal to find optimal paths efficiently.'), 0.5809713116576584), (Document(id='Artificial_Intelligence__grades__p0__c00003', metadata={'file_name': 'grades.csv', 'course':

### Answer
The A* search algorithm combines the cost so far with a heuristic estimate of the cost to the goal to find optimal paths efficiently.

*Moderate confidence - top match similarity 0.58.*

### Sources
1. **Artificial Intelligence / search_algorithms.pdf, page 1** (chunk `Artificial_Intelligence__search_algorithms__p1__c00005`, similarity `0.581`)
2. **Artificial Intelligence / grades.csv** (chunk `Artificial_Intelligence__grades__p0__c00003`, similarity `0.0556`)
3. **Artificial Intelligence / intro_to_ai.pdf, page 1** (chunk `Artificial_Intelligence__intro_to_ai__p1__c00004`, similarity `-0.0381`)
4. **Artificial Intelligence / glossary.txt** (chunk `Artificial_Intelligence__glossary__p0__c00002`, similarity `-0.0683`)
2026-08-02 01:04:58 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.


f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Deep_Learning__neural_networks_basics__p1__c00009', metadata={'course': 'Deep Learning', 'chunk_id': 'Deep_Learning__neural_networks_basics__p1__c00009', 'page_number': 1, 'char_count': 440, 'doc_type': 'pdf', 'file_name': 'neural_networks_basics.pdf'}, page_content='Lecture 1: Neural Network Fundamentals\nA neural network is composed of layers of interconnected nodes, or neurons, each applying a weighted\nsum followed by a non-linear activation function such as ReLU, Sigmoid, or Tanh.\nBackpropagation\nBackpropagation is the algorithm used to compute gradients of the loss function with respect to each\nweight by applying the chain rule, allowing the network to update its parameters via gradient descent.'), 0.6834987679430478), (Document(id='Deep_Learning__glossary__p0__c00007', metadata={'page_number': -1, 'course': 'Deep Lear

### Answer
Backpropagation is the algorithm used to compute gradients of the loss function with respect to each weight by applying the chain rule, allowing the network to update its parameters via gradient descent.

*Moderate confidence - top match similarity 0.68.*

### Sources
1. **Deep Learning / neural_networks_basics.pdf, page 1** (chunk `Deep_Learning__neural_networks_basics__p1__c00009`, similarity `0.6835`)
2. **Deep Learning / glossary.txt** (chunk `Deep_Learning__glossary__p0__c00007`, similarity `0.2352`)
3. **Deep Learning / cnn_architectures.docx** (chunk `Deep_Learning__cnn_architectures__p0__c00006`, similarity `0.0826`)
4. **Machine Learning / supervised_learning.docx** (chunk `Machine_Learning__supervised_learning__p0__c00012`, similarity `-0.0012`)
2026-08-02 01:05:07 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.
### Answer
Dropout is a regularization technique used in deep learning that randomly deactivates a fraction of neurons during training to re

## Source Attribution

Every answer carries its supporting sources: course name, file name, page
number (when available), chunk ID, and similarity score.

In [9]:
last_result = results[-1]

for row in AnswerFormatter.to_source_table(last_result):
    course, file_name, page, chunk_id, score = row
    print(f"{course:<25} {file_name:<28} page={page!s:<5} score={score:<6} chunk_id={chunk_id}")


Deep Learning             glossary.txt                 page=-     score=0.5512 chunk_id=Deep_Learning__glossary__p0__c00007
Deep Learning             cnn_architectures.docx       page=-     score=0.1743 chunk_id=Deep_Learning__cnn_architectures__p0__c00006
Machine Learning          glossary.txt                 page=-     score=0.108  chunk_id=Machine_Learning__glossary__p0__c00010
Deep Learning             neural_networks_basics.pdf   page=1     score=0.0646 chunk_id=Deep_Learning__neural_networks_basics__p1__c00009


## Evaluation

We check three properties over the example questions:

1. Every answered question returned at least one source.
2. No answer is empty.
3. The confidence note is present for every response.

In [10]:
all_have_sources = all(len(r.sources) > 0 for r in results)
all_non_empty = all(bool(r.answer.strip()) for r in results)
all_have_confidence = all(bool(r.confidence_note) for r in results)

print("All answers have >=1 source :", all_have_sources)
print("All answers are non-empty   :", all_non_empty)
print("All answers have a confidence note:", all_have_confidence)

avg_time = sum(r.generation_seconds for r in results) / len(results)
print(f"Average generation time: {avg_time:.2f}s")


All answers have >=1 source : True
All answers are non-empty   : True
All answers have a confidence note: True
Average generation time: 8.46s


## Failure Cases

Two deliberately adversarial questions:

1. A question with **no relevant course material** — the system should
   respond with the fixed "not found" message instead of hallucinating.
2. A question **outside the scope** of any of the three courses.

In [11]:
out_of_scope_result = ask("What is the boiling point of mercury at sea level?")
print("\n--- error field ---")
print(out_of_scope_result.error)


2026-08-02 01:05:15 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.


f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Artificial_Intelligence__grades__p0__c00003', metadata={'doc_type': 'csv', 'file_name': 'grades.csv', 'char_count': 482, 'page_number': -1, 'course': 'Artificial Intelligence', 'chunk_id': 'Artificial_Intelligence__grades__p0__c00003'}, page_content='StudentID, Assignment, Score, MaxScore\nStudentID: 101; Assignment: Search Algorithms Homework; Score: 88; MaxScore: 100\nStudentID: 102; Assignment: Search Algorithms Homework; Score: 92; MaxScore: 100\nStudentID: 103; Assignment: Search Algorithms Homework; Score: 75; MaxScore: 100\nStudentID: 101; Assignment: Midterm Exam; Score: 81; MaxScore: 100\nStudentID: 102; Assignment: Midterm Exam; Score: 90; MaxScore: 100\nStudentID: 103; Assignment: Midterm Exam; Score: 70; MaxScore: 100'), -0.14777281861741187), (Document(id='Machine_Learning__grades__p0__c00011', metadata={'page_numb

### Answer
I couldn't find this information in the provided course materials.

*Low confidence - top match similarity -0.15. Answer may be incomplete.*

### Sources
1. **Artificial Intelligence / grades.csv** (chunk `Artificial_Intelligence__grades__p0__c00003`, similarity `-0.1478`)
2. **Machine Learning / grades.csv** (chunk `Machine_Learning__grades__p0__c00011`, similarity `-0.2057`)
3. **Machine Learning / supervised_learning.docx** (chunk `Machine_Learning__supervised_learning__p0__c00012`, similarity `-0.3297`)
4. **Machine Learning / glossary.txt** (chunk `Machine_Learning__glossary__p0__c00010`, similarity `-0.3618`)

--- error field ---
None


In [12]:
unrelated_result = ask("Who won the FIFA World Cup in 2018?")


2026-08-02 01:05:21 | INFO     | RetrieverEngine | Retrieved 4 chunk(s) for query.


f:\courses\Ai Instant\Revision\Sprint_4\Course_Q&A_Assistant\rag_core\retriever.py:51: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='Deep_Learning__grades__p0__c00008', metadata={'course': 'Deep Learning', 'page_number': -1, 'chunk_id': 'Deep_Learning__grades__p0__c00008', 'file_name': 'grades.csv', 'char_count': 476, 'doc_type': 'csv'}, page_content='StudentID, Assignment, Score, MaxScore\nStudentID: 301; Assignment: CNN Image Classifier; Score: 90; MaxScore: 100\nStudentID: 302; Assignment: CNN Image Classifier; Score: 86; MaxScore: 100\nStudentID: 303; Assignment: CNN Image Classifier; Score: 93; MaxScore: 100\nStudentID: 301; Assignment: RNN Sequence Lab; Score: 82; MaxScore: 100\nStudentID: 302; Assignment: RNN Sequence Lab; Score: 88; MaxScore: 100\nStudentID: 303; Assignment: RNN Sequence Lab; Score: 79; MaxScore: 100'), -0.1656499944691867), (Document(id='Machine_Learning__grades__p0__c00011', metadata={'course': 'Machine Learning', 'page_number': -1

### Answer
I couldn't find this information in the provided course materials.

*Low confidence - top match similarity -0.17. Answer may be incomplete.*

### Sources
1. **Deep Learning / grades.csv** (chunk `Deep_Learning__grades__p0__c00008`, similarity `-0.1656`)
2. **Machine Learning / grades.csv** (chunk `Machine_Learning__grades__p0__c00011`, similarity `-0.2522`)
3. **Artificial Intelligence / grades.csv** (chunk `Artificial_Intelligence__grades__p0__c00003`, similarity `-0.2717`)
4. **Machine Learning / supervised_learning.docx** (chunk `Machine_Learning__supervised_learning__p0__c00012`, similarity `-0.2869`)


## Deployment Preparation

The same `AppConfig`, `EmbeddingManager`, `VectorDatabaseManager`,
`RetrieverEngine`, `PromptManager`, `RAGPipeline`, and `AnswerFormatter`
classes used in this notebook are imported directly by `app.py`, so no
logic needs to be duplicated or rewritten for the Gradio deployment.

In [13]:
print("Deployment checklist:")
print(" - chroma_db/ contains a persisted collection:", vector_db_manager.exists())
print(" - rag_core package importable from project root:", (PROJECT_ROOT / "rag_core").exists())
print(" - app.py present:", (PROJECT_ROOT / "app.py").exists())
print("\nRun the app locally with:  python app.py")


Deployment checklist:
 - chroma_db/ contains a persisted collection: True
 - rag_core package importable from project root: True
 - app.py present: True

Run the app locally with:  python app.py
